In [9]:
!pip install gspread-dataframe gspread-formatting

import pandas as pd
import numpy as np
import gspread
import re
import unicodedata
from gspread_dataframe import set_with_dataframe
import gspread_formatting as gsf
from google.colab import auth
from google.auth import default
from datetime import datetime, timedelta

# ==========================================
# 1. AUTENTICAÇÃO E CONEXÃO
# ==========================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# COLOQUE O LINK DA SUA PLANILHA AQUI
planilha_url = 'https://docs.google.com/spreadsheets/d/1zEn8jrIBOIQwkHmQQ880v8SUpFO5sONHFQ7LWpwH9PM/edit?gid=558033855#gid=558033855'
planilha = gc.open_by_url(planilha_url)

# ==========================================
# 2. FUNÇÕES DE APOIO E REGRAS DE NEGÓCIO
# ==========================================
def limpar_texto(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return "sem comentário"
    texto = "".join(c for c in unicodedata.normalize('NFD', str(texto)) if unicodedata.category(c) != 'Mn')
    return texto.lower().strip()

def filtrar_ruido(df):
    return df[ (df['Avaliação'].str.len() >= 4) | (df['Avaliação'] == "sem comentário") ].copy()

def classificar_smart(comentario):
    comentario = limpar_texto(comentario)
    regras = {
        'qualidade ruim': ['fragil', 'ruim', 'pessim', 'material', 'quebrou', 'descosturou', 'fraco', 'fina'],
        'tamanho incorreto': ['pequeno', 'grande', 'apertado', 'largo', 'forma', 'numero', 'medida', 'curto'],
        'diferente do anunciado': ['diferente', 'foto', 'engano', 'outra cor', 'nao e igual'],
        'produto falso': ['falso', 'replica', 'original', 'pirata', 'falsificado'],
        'defeito': ['defeito', 'parou', 'nao liga', 'quebrado', 'riscado', 'estragado', 'rasgado'],
        'entrega': ['atraso', 'demorou', 'prazo', 'transportadora', 'correio'],
        'não recebido': ['nao recebi', 'nao chegou', 'entregue mas nao recebi', 'cade']
    }
    for motivo, palavras in regras.items():
        if any(p in comentario for p in palavras):
            return motivo
    return 'outros'

def obter_resumo_motivo(series):
    motivos_reais = series[~series.isin(['sem comentário', 'outros'])]
    if not motivos_reais.empty:
        return motivos_reais.mode()[0]
    elif not series.empty:
        return series.mode()[0]
    return "sem dados"

def classificar_status(vol, media):
    if vol < 7:
        return 'Baixo numero de avaliações'
    elif media < 2:
        return 'Ruim'
    elif media < 3:
        return 'Atenção'
    else:
        return 'Bons'

def definir_prioridade(vol, media):
    if vol >= 25 and media < 1.8:
        return "1 - CRÍTICA 🚨"
    elif vol >= 15 and media < 2.2:
        return "2 - ALTA 🔥"
    elif vol >= 7 and media < 2.5:
        return "3 - MÉDIA ⚡"
    else:
        return "4 - BAIXA 🟢"

def calcular_variacao(x):
    if pd.isna(x): return "Sem base ant."
    return f"{x:.2f}"

# ==========================================
# 3. LEITURA E TRATAMENTO
# ==========================================
aba_base = planilha.worksheet('Base de reviews')
df = pd.DataFrame(aba_base.get_all_records())

df['Avaliação_Limpa'] = df['Avaliação'].apply(limpar_texto)
df = filtrar_ruido(df)
df['Data da avaliação'] = pd.to_datetime(df['Data da avaliação'], errors='coerce')
df['Nota'] = pd.to_numeric(df['Nota'], errors='coerce')
df['is_recomendado'] = np.where(df['Campo - Você recomendaria esse produto?'].astype(str).str.lower().str.contains('sim'), 1, 0)
df['Motivo'] = df['Avaliação'].apply(classificar_smart)
df['Mes'] = df['Data da avaliação'].dt.to_period('M')

# ==========================================
# 4. PROCESSAMENTO DAS MÉTRICAS (MoM)
# ==========================================
mes_atual = df['Mes'].max()
mes_anterior = mes_atual - 1

def get_mom_metrics(df, group_col):
    atual = df[df['Mes'] == mes_atual].groupby(group_col)['Nota'].mean()
    anterior = df[df['Mes'] == mes_anterior].groupby(group_col)['Nota'].mean()
    mom = (atual - anterior).apply(calcular_variacao).rename('Variação Mês (MoM)')
    return mom

mom_prod = get_mom_metrics(df, 'ID Produto')
mom_seller = get_mom_metrics(df, 'Seller')

# ==========================================
# 5. CONSOLIDAÇÃO DAS VISÕES
# ==========================================

# PRODUTOS
visao_produto = df.groupby(['ID Produto', 'Produto']).agg(
    Qtd_Notas=('Nota', 'count'),
    Media_Nota=('Nota', 'mean'),
    Perc_Recomendacao=('is_recomendado', lambda x: f"{(x.mean() * 100):.1f}%"),
    Resumo_Avaliacao=('Motivo', obter_resumo_motivo)
).reset_index()
visao_produto = visao_produto.merge(mom_prod, on='ID Produto', how='left')
visao_produto['Status'] = visao_produto.apply(lambda x: classificar_status(x['Qtd_Notas'], x['Media_Nota']), axis=1)
visao_produto['Media_Nota'] = visao_produto['Media_Nota'].round(2)

# SELLERS (Adicionado o Id da Loja na agregação)
visao_seller = df.groupby('Seller').agg(
    Id_da_Loja=('Id da Loja', 'first'),  # Puxa o ID da loja correspondente ao Seller
    Qtd_Notas=('Nota', 'count'),
    Media_Nota=('Nota', 'mean'),
    Perc_Recomendacao=('is_recomendado', lambda x: f"{(x.mean() * 100):.1f}%"),
    Resumo_Avaliacao=('Motivo', obter_resumo_motivo)
).reset_index()
visao_seller = visao_seller.merge(mom_seller, on='Seller', how='left')
visao_seller['Status'] = visao_seller.apply(lambda x: classificar_status(x['Qtd_Notas'], x['Media_Nota']), axis=1)
visao_seller['Media_Nota'] = visao_seller['Media_Nota'].round(2)

# ==========================================
# 6. CRIAÇÃO DA ABA TAREFAS
# ==========================================
tarefas_lista = []

# Adicionando Produtos (com ID Produto)
df_criticos_prod = visao_produto[visao_produto['Status'].isin(['Ruim', 'Atenção'])].copy()
for _, row in df_criticos_prod.iterrows():
    tarefas_lista.append({
        'Prioridade': definir_prioridade(row['Qtd_Notas'], row['Media_Nota']),
        'Tarefa': 'Revisão de Conteúdo/Qualidade Produto',
        'Produto ou Seller': f"PRODUTO: {row['Produto']}",
        'ID (Produto/Loja)': row['ID Produto'], # <--- NOVA COLUNA
        'Qtd de Notas': row['Qtd_Notas'],
        'Média de Nota': row['Media_Nota'],
        'Perc_Recomendacao': row['Perc_Recomendacao'],
        'Resumo_da_avaliacao': row['Resumo_Avaliacao'],
        'Variação Mês (MoM)': row['Variação Mês (MoM)'],
        'Status': row['Status']
    })

# Adicionando Sellers (com Id da Loja)
df_criticos_seller = visao_seller[visao_seller['Status'].isin(['Ruim', 'Atenção'])].copy()
for _, row in df_criticos_seller.iterrows():
    tarefas_lista.append({
        'Prioridade': definir_prioridade(row['Qtd_Notas'], row['Media_Nota']),
        'Tarefa': 'Abrir Plano de Ação com Seller',
        'Produto ou Seller': f"SELLER: {row['Seller']}",
        'ID (Produto/Loja)': row['Id_da_Loja'], # <--- NOVA COLUNA
        'Qtd de Notas': row['Qtd_Notas'],
        'Média de Nota': row['Media_Nota'],
        'Perc_Recomendacao': row['Perc_Recomendacao'],
        'Resumo_da_avaliacao': row['Resumo_Avaliacao'],
        'Variação Mês (MoM)': row['Variação Mês (MoM)'],
        'Status': row['Status']
    })

df_tarefas_v3 = pd.DataFrame(tarefas_lista)
if not df_tarefas_v3.empty:
    df_tarefas_v3 = df_tarefas_v3.sort_values(by=['Prioridade', 'Qtd de Notas'], ascending=[True, False])

# ==========================================
# 7. EXPORTAÇÃO E FORMATAÇÃO
# ==========================================
def atualizar_e_formatar(nome_aba, dataframe):
    try:
        aba = planilha.worksheet(nome_aba)
        aba.clear()
    except gspread.exceptions.WorksheetNotFound:
        aba = planilha.add_worksheet(title=nome_aba, rows="1000", cols="20")

    set_with_dataframe(aba, dataframe)
    gsf.format_cell_range(aba, 'A1:Z1', gsf.cellFormat(textFormat=gsf.textFormat(bold=True), backgroundColor=gsf.color(0.9, 0.9, 0.9)))

    rules = gsf.get_conditional_format_rules(aba)
    rules.clear()

    if nome_aba == 'Tarefas':
        rules.append(gsf.ConditionalFormatRule(
            ranges=[gsf.GridRange.from_a1_range('A2:Z1000', aba)],
            booleanRule=gsf.BooleanRule(condition=gsf.BooleanCondition('TEXT_CONTAINS', ['CRÍTICA']), format=gsf.cellFormat(backgroundColor=gsf.color(1.0, 0.7, 0.7)))
        ))
        rules.append(gsf.ConditionalFormatRule(
            ranges=[gsf.GridRange.from_a1_range('A2:Z1000', aba)],
            booleanRule=gsf.BooleanRule(condition=gsf.BooleanCondition('TEXT_CONTAINS', ['ALTA']), format=gsf.cellFormat(backgroundColor=gsf.color(1.0, 0.9, 0.6)))
        ))

    rules.save()

# Enviando para o Sheets
atualizar_e_formatar('Visao_Produto', visao_produto.sort_values(by='Qtd_Notas', ascending=False))
atualizar_e_formatar('Visao_Seller', visao_seller.drop(columns=['Id_da_Loja']).sort_values(by='Qtd_Notas', ascending=False)) # Esconde o ID da aba do Seller para manter limpa
atualizar_e_formatar('Tarefas', df_tarefas_v3)

print("🚀 Pipeline atualizado! A coluna 'ID (Produto/Loja)' foi incluída com sucesso nas Tarefas.")

🚀 Pipeline atualizado! A coluna 'ID (Produto/Loja)' foi incluída com sucesso nas Tarefas.
